# Omni-fMRI Quick Start: Feature Extraction

This notebook mirrors the project-page Quick Start. It creates a tiny synthetic fMRI NIfTI, runs the official preprocessing script, extracts Omni-fMRI features with the released checkpoint, and checks the output arrays.

## 1. Clone and install

```bash
git clone https://github.com/OneMore1/Omni-fMRI.git
cd Omni-fMRI
conda create -n omnifmri python=3.11 -y
conda activate omnifmri
pip install -r requirements.txt
```

Download the Hugging Face checkpoint from `https://huggingface.co/OneMore1/Omni-fMRI` and place it at `pretrain_checkpoint/checkpoint.pth`.

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib

work_dir = Path('outputs/quickstart_notebook')
raw_dir = work_dir / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(7)
data = np.zeros((96, 96, 96, 40), dtype=np.float32)
data[32:64, 30:66, 28:70, :] = rng.normal(0, 0.1, size=(32, 36, 42, 40)).astype(np.float32)
data[46:52, 44:54, 40:56, :] += np.linspace(-0.5, 0.5, 40, dtype=np.float32)

raw_path = raw_dir / 'sample_fmri.nii.gz'
nib.save(nib.Nifti1Image(data, affine=np.eye(4)), raw_path)
raw_path

In [ ]:
!python data_preparation/preprocessing.py \
  -i outputs/quickstart_notebook/raw \
  -o outputs/quickstart_notebook/processed_npz \
  --pattern .nii.gz \
  --target_shape 96 96 96 \
  -s 40

In [ ]:
!python extract_feat.py \
  outputs/quickstart_notebook/processed_npz \
  --checkpoint pretrain_checkpoint/checkpoint.pth \
  --output-dir outputs/quickstart_notebook/features \
  --device cuda:0 \
  --overwrite

In [ ]:
outputs = sorted((work_dir / 'features').glob('*_tokens.npz'))
assert outputs, 'No feature files were produced.'
tokens = np.load(outputs[0])
for key in ['cls_token', 'patch_tokens', 'patch_coords']:
    print(key, tokens[key].shape, tokens[key].dtype)